In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv('../data/processed/dataset_final.csv')
indec = pd.read_csv('../data/processed/indec_limpio.csv', dtype={'periodo': str})

print(f"Dataset final: {df.shape[0]} filas | {df.shape[1]} columnas")
print("Listos para explorar")

Dataset final: 3676 filas | 30 columnas
Listos para explorar


In [2]:
print("=== ESTADÍSTICAS GENERALES ===")
print(f"Total profesionales:        {len(df):,}")
print(f"Salario bruto promedio:     ${df['salario_bruto'].mean():>15,.0f} ARS")
print(f"Salario bruto mediana:      ${df['salario_bruto'].median():>15,.0f} ARS")
print(f"Salario USD promedio:       USD {df['salario_usd_mensual'].mean():>8,.0f}/mes")
print(f"Salario USD mediana:        USD {df['salario_usd_mensual'].median():>8,.0f}/mes")
print(f"Salario real 2020 promedio: ${df['salario_real_2020'].mean():>15,.0f} ARS")
print(f"\nModalidades:")
print(df['modalidad'].value_counts())
print(f"\nSeniority:")
print(df['seniority'].value_counts())

=== ESTADÍSTICAS GENERALES ===
Total profesionales:        3,676
Salario bruto promedio:     $      3,265,023 ARS
Salario bruto mediana:      $      2,900,000 ARS
Salario USD promedio:       USD    2,433/mes
Salario USD mediana:        USD    2,161/mes
Salario real 2020 promedio: $        102,934 ARS

Modalidades:
modalidad
100% remoto                      1811
híbrido (presencial y remoto)    1586
100% presencial                   279
Name: count, dtype: int64

Seniority:
seniority
senior         2029
semi-senior    1198
junior          449
Name: count, dtype: int64


In [3]:
fig = px.histogram(
    df,
    x='salario_usd_mensual',
    nbins=50,
    title='Distribución de Salarios Tech en Argentina (USD/mes)',
    labels={'salario_usd_mensual': 'Salario USD/mes'},
    color_discrete_sequence=['#2563eb']
)
fig.add_vline(x=df['salario_usd_mensual'].median(), line_dash='dash', 
              line_color='red', annotation_text=f"Mediana: USD {df['salario_usd_mensual'].median():,.0f}")
fig.update_layout(showlegend=False)
fig.show()

In [4]:
orden = ['junior', 'semi-senior', 'senior']

fig = px.box(
    df,
    x='seniority',
    y='salario_usd_mensual',
    category_orders={'seniority': orden},
    title='Salario USD por Seniority',
    labels={'seniority': 'Seniority', 'salario_usd_mensual': 'Salario USD/mes'},
    color='seniority',
    color_discrete_sequence=['#93c5fd', '#2563eb', '#1e3a8a']
)
fig.show()

In [5]:
mediana_modalidad = df.groupby('modalidad')['salario_usd_mensual'].median().reset_index()
mediana_modalidad = mediana_modalidad.sort_values('salario_usd_mensual', ascending=True)

fig = px.bar(
    mediana_modalidad,
    x='salario_usd_mensual',
    y='modalidad',
    orientation='h',
    title='Salario Mediano USD por Modalidad de Trabajo',
    labels={'salario_usd_mensual': 'Salario USD/mes', 'modalidad': ''},
    color='salario_usd_mensual',
    color_continuous_scale='Blues'
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [9]:
### Celda 6 corregida — IPC con etiquetas limpias
indec['año'] = indec['periodo'].str[:4]
indec['mes'] = indec['periodo'].str[4:]
indec['fecha_label'] = indec['año'] + '-' + indec['mes']

# Mostrar solo etiquetas de enero de cada año
tickvals = indec[indec['mes'] == '01']['periodo'].tolist()
ticktext = indec[indec['mes'] == '01']['año'].tolist()

fig = px.line(
    indec,
    x='periodo',
    y='indice_ipc',
    title='Evolución IPC Nacional 2020-2026 (Base: Dic 2016 = 100)',
    labels={'periodo': 'Año', 'indice_ipc': 'Índice IPC'},
    color_discrete_sequence=['#dc2626']
)
fig.update_layout(
    xaxis=dict(tickvals=tickvals, ticktext=ticktext)
)
fig.show()

In [7]:
brecha = df.groupby('rol_estandar').agg(
    argentina=('salario_usd_anual', 'median'),
    latam=('salario_usd_anual_latam', 'first')
).dropna().reset_index()

brecha_melted = brecha.melt(
    id_vars='rol_estandar', 
    value_vars=['argentina', 'latam'],
    var_name='mercado', 
    value_name='salario_usd_anual'
)

fig = px.bar(
    brecha_melted,
    x='salario_usd_anual',
    y='rol_estandar',
    color='mercado',
    barmode='group',
    orientation='h',
    title='Salario Anual USD — Argentina vs LATAM por Rol',
    labels={'salario_usd_anual': 'Salario USD/año', 'rol_estandar': '', 'mercado': 'Mercado'},
    color_discrete_map={'argentina': '#2563eb', 'latam': '#93c5fd'}
)
fig.show()

In [14]:
### Celda 8 corregida — Filtrar valores no tecnológicos
excluir = ['ninguno de los anteriores', 'no utilizo', 'ninguna', 'otros', 'other']

tecnologias_exploded = df['tecnologias'].dropna().str.lower().str.split(',').explode()
tecnologias_exploded = tecnologias_exploded.str.strip()
tecnologias_exploded = tecnologias_exploded[~tecnologias_exploded.isin(excluir)]

top_techs = tecnologias_exploded.value_counts().head(15).reset_index()
top_techs.columns = ['tecnologia', 'cantidad']

fig = px.bar(
    top_techs,
    x='cantidad',
    y='tecnologia',
    orientation='h',
    title='Top 15 Tecnologías más Usadas en el Mercado Tech Argentino',
    labels={'cantidad': 'Cantidad de profesionales', 'tecnologia': ''},
    color='cantidad',
    color_continuous_scale='Blues'
)
fig.update_layout(coloraxis_showscale=False, yaxis={'categoryorder': 'total ascending'})
fig.show()

In [11]:
# Para cada profesional, expandimos sus tecnologías y le asignamos su salario
df_tech = df[['tecnologias', 'salario_usd_mensual']].copy()
df_tech = df_tech.dropna(subset=['tecnologias'])
df_tech['tecnologia'] = df_tech['tecnologias'].str.lower().str.split(',')
df_tech = df_tech.explode('tecnologia')
df_tech['tecnologia'] = df_tech['tecnologia'].str.strip()

# Filtrar tecnologías con al menos 30 respuestas para que sea representativo
conteo = df_tech['tecnologia'].value_counts()
techs_validas = conteo[conteo >= 30].index

salario_por_tech = df_tech[df_tech['tecnologia'].isin(techs_validas)].groupby('tecnologia')['salario_usd_mensual'].median().reset_index()
salario_por_tech.columns = ['tecnologia', 'salario_mediano_usd']
salario_por_tech = salario_por_tech.sort_values('salario_mediano_usd', ascending=True).tail(15)

fig = px.bar(
    salario_por_tech,
    x='salario_mediano_usd',
    y='tecnologia',
    orientation='h',
    title='Top 15 Tecnologías por Salario Mediano (USD/mes)',
    labels={'salario_mediano_usd': 'Salario Mediano USD/mes', 'tecnologia': ''},
    color='salario_mediano_usd',
    color_continuous_scale='Blues'
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [12]:
top_provincias = df['provincia'].value_counts().head(8).index

df_prov = df[df['provincia'].isin(top_provincias)].copy()

mediana_prov = df_prov.groupby('provincia')['salario_usd_mensual'].median().reset_index()
mediana_prov = mediana_prov.sort_values('salario_usd_mensual', ascending=True)

fig = px.bar(
    mediana_prov,
    x='salario_usd_mensual',
    y='provincia',
    orientation='h',
    title='Salario Mediano USD por Provincia (Top 8)',
    labels={'salario_usd_mensual': 'Salario USD/mes', 'provincia': ''},
    color='salario_usd_mensual',
    color_continuous_scale='Blues'
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [13]:
cols_numericas = [
    'salario_usd_mensual', 'anos_experiencia', 
    'antiguedad_empresa', 'anos_puesto'
]

# Convertir a numérico por si quedaron como string
for col in cols_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce')

correlaciones = df[cols_numericas].corr().round(2)

fig = px.imshow(
    correlaciones,
    text_auto=True,
    color_continuous_scale='Blues',
    title='Correlación entre Variables Numéricas',
    labels=dict(color='Correlación')
)
fig.show()